# DROUGHTS

## INTRODUCCIÓN Y OBJETIVOS

Objetivo del estudio

El objetivo de este análisis es estudiar la relación entre los episodios de calor extremo y los periodos de déficit de precipitaciones con la disponibilidad de recursos hídricos en Cataluña.

Para ello, se utilizarán datos meteorológicos de Meteocat (XEMA) y datos hidrológicos de la Agència Catalana de l'Aigua (ACA), incluyendo información sobre las reservas de los embalses y los estados oficiales de sequía.

Se pretende:
- Identificar episodios de temperaturas extremas y déficit de precipitaciones.
- Analizar la evolución temporal de las reservas de agua.
- Estudiar la relación entre las condiciones meteorológicas y la disponibilidad de agua.
- Comparar los indicadores meteorológicos e hidrológicos con los estados oficiales de sequía.
- Identificar posibles retardos temporales entre la falta de precipitaciones y sus efectos sobre los recursos hídricos.

## IMPORTACIÓN LIBRERÍAS

In [3]:
# Data manipulation
import pandas as pd
import numpy as np

# Data downloading
import requests

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Date handling
from datetime import datetime

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid")


## CARGAR LOS DATOS DE LA XEMA

In [4]:
# XEMA - Meteorological daily data

path_xema = r"C:\Users\nmayo\OneDrive\Escritorio\Máster - UB\Análisis de datos masivos\Datasets - Agua\Dades_meteorològiques_diàries_de_la_XEMA_20261116.csv"
df_xema = pd.read_csv(path_xema, low_memory=False)

print("XEMA loaded successfully")
print("Dimensions:", df_xema.shape)

display(df_xema.head())

XEMA loaded successfully
Dimensions: (33475032, 10)


,ID,CODI_ESTACIO,NOM_ESTACIO,DATA_LECTURA,CODI_VARIABLE,NOM_VARIABLE,VALOR,UNITAT,HORA _TU,Estat
0,VK1000098801,VK,Raimat,01/09/1988,1.00,Temperatura mitjana diària,"23,3",°C,NaN,Representatiu
1,VK1002098801,VK,Raimat,01/09/1988,1.00,Temperatura mínima diària + hora,"18,3",°C,0:00,Representatiu
2,VK1100098801,VK,Raimat,01/09/1988,1.10,Humitat relativa mitjana diària,58,%,NaN,Representatiu
3,VK1300098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària,"0,6",mm,NaN,Representatiu
4,VK1301098801,VK,Raimat,01/09/1988,1.30,Precipitació acumulada diària (8-8 h),"0,6",mm,NaN,Representatiu


## CARGA DATOS DE LOS EMBALSES DE LA ACA

In [5]:
# ACA - Water quantity in reservoirs

url_aca = "https://analisi.transparenciacatalunya.cat/api/v3/views/gn9e-3qhr/query.json"

def download_socrata(url, limit=50000):
    frames = []
    offset = 0

    while True:
        params = {
            "$limit": limit,
            "$offset": offset,
            "$order": "dia ASC"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        batch = response.json()

        if not batch:
            break

        frames.append(pd.DataFrame(batch))
        offset += len(batch)

        if len(batch) < limit:
            break

    return pd.concat(frames, ignore_index=True)

df_aca = download_socrata(url_aca)

print("ACA reservoirs loaded successfully")
print("Dimensions:", df_aca.shape)

display(df_aca.head())

KeyboardInterrupt: 

## DESCARGAR DATOS DE SEQUÍA

In [ ]:
url_sequera = "https://analisi.transparenciacatalunya.cat/api/v3/views/i5n8-43cw/query.json"

df_sequera = download_socrata(url_sequera)

print("Drought dataset loaded successfully")
print("Dimensions:", df_sequera.shape)

display(df_sequera.head())

## EXPLORACIÓN DE LOS DATASETS

### CONSULTAR COLUMNAS DISPONIBLES

In [ ]:
datasets = {
    "XEMA": df_xema,
    "ACA reservoirs": df_aca,
    "ACA drought": df_sequera
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")

    print("Dimensions:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

### CONSULTAR TIPOS DE DATOS

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")

    df.info()

### COMPROBAR LOS VALORES NULOS

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")

    missing = df.isnull().sum()

    missing_percentage = (missing / len(df)) * 100

    missing_table = pd.DataFrame({
        "Missing values": missing,
        "Percentage (%)": missing_percentage
    })

    display(missing_table)

### COMPROBAR LOS DUPLICADOS

In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicated rows")